## **1. DATA DESCRIPTION**

## **1.1 Prices**

1. **Temporal Scope** 

    This dataset contains historical electricity prices from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers 20 bidding zones, which can be grouped into three larger regions:
    -  <u>Baltic countries </u> 

        Lithuania, Latvia and Estonia

    -  <u>Nordic countries </u> 
        
        Denmark, Finland, Norway and Sweden

    -  <u>Central Western Europe </u> 
        
        Germany, France, the Netherlands, Belgium and Austria

    Denmark, Norway and Sweden are divided into several bidding zones.

    Other bidding zones are only present during a limited period of time and they are excluded from the analysis to ensure consistency in the geographical coverage.

    - Finland-Russia Exchange (01/01/2020 - 03/06/2020)
    - Great Britain (01/01/2020 - 31/12/2020)
    - Bulgaria (02/12/2020 - present)
    - Romania (20/11/2024 - present)

3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    EUR/MWh

6. **Structure**

    The dataset is stored in a wide format, where each row represents a delivery period and each bidding zone is represented as a separate column.

7. **Role in the analysis**

    This dataset represents the main target variable of the study, as it contains the electricity prices to be analyzed and modeled across time and bidding zones.


## **1.2 Volumes**

1. **Temporal Scope** 

    This dataset contains historical electricity market volumes from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers 20 bidding zones, which can be grouped into three larger regions:
    -  <u>Baltic countries </u> 

        Lithuania, Latvia and Estonia

    -  <u>Nordic countries </u> 
        
        Denmark, Finland, Norway and Sweden

    -  <u>Central Western Europe </u> 
        
        Germany, France, the Netherlands, Belgium and Austria

    Denmark, Norway and Sweden are divided into several bidding zones.

    Other bidding zones are only present during a limited period of time and they are excluded from the analysis to ensure consistency in the geographical coverage.


3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    MWh to ensure consistency with the price dataset, which is expressed in EUR/MWh.

6. **Structure**

    The dataset is stored in a wide format. For each bidding zone, two consecutive columns are provided, corresponding to Buy and Sell volumes.

7. **Role in the analysis**

    This dataset represents market activity in each bidding zone, reflecting supply (sell volumes) and demand (buy volumes), and may help explain electricity price dynamics.

## **1.3 Flows**

1. **Temporal Scope** 

    This dataset contains historical directional energy flows from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers directional interconnections between bidding zones included in the analysis. Each extracted table is centered on a specific delivery area and contains the corresponding import and export flows with neighboring zones.


3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    MWh

6. **Structure**

    The dataset is stored in a wide format, where each row represents a delivery period and each column represents a directional interconnection between two areas (e.g. NO1 -> NO2, NO2 -> NO1).

7. **Role in the analysis**

    This dataset captures cross-zonal energy exchanges and provides information about the spatial interaction between bidding zones. It may help explain price differences, market coupling, and network-related events.

## **1.4 Capacities**

1. **Temporal Scope** 

    This dataset contains historical transmission capacities from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers directional transmission capacities between bidding zones included in the analysis.

    The raw data may include additional technical nodes (e.g. sub-areas or interconnectors), which are excluded to ensure consistency with the bidding zones used in the rest of the study.


3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    MWh

6. **Structure**

    The dataset is stored in a wide format, where each row represents a delivery period and each column represents a directional capacity between two areas (e.g. NO1 → NO2, NO2 → NO1).

7. **Role in the analysis**

    This dataset represents the physical limits of the transmission network and provides information about the maximum possible energy exchange between zones. It is essential to identify potential congestion situations and to contextualize observed flows.

## **2. DATA LOADING**

## **2.1 Prices**

In this section, the downloaded Excel files corresponding to the Prices dataset are loaded and combined into a single raw dataframe. At this stage, no preprocessing is applied yet.

In [46]:
import os
import glob
import pandas as pd

In [47]:
# Base path for the downloaded Prices files
prices_base_path = r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\NordPoool\ExcelFilesNoProcessed\Prices"

# Read all Excel files from subfolders (e.g. 2020, 2021, ...)
price_files_2020 = glob.glob(os.path.join(prices_base_path, "2020", "*.xlsx"))
price_files_2021 = glob.glob(os.path.join(prices_base_path, "2021", "*.xlsx"))

price_files = sorted(price_files_2020 + price_files_2021)

print(f"2020 files found: {len(price_files_2020)}")
print(f"2021 files found: {len(price_files_2021)}")
print(f"Total files found: {len(price_files)}")

2020 files found: 366
2021 files found: 365
Total files found: 731


Track if every price file has been downloaded and loaded correctly.

In [48]:
import os
import re
import pandas as pd

path_2021 = r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\NordPoool\ExcelFilesNoProcessed\Prices\2021"

files = [f for f in os.listdir(path_2021) if f.endswith(".xlsx")]

dates_found = []
for f in files:
    match = re.search(r"\d{4}-\d{2}-\d{2}", f)
    if match:
        dates_found.append(match.group())

dates_found = pd.to_datetime(dates_found)
expected_dates = pd.date_range("2021-01-01", "2021-12-31")

missing_dates = expected_dates.difference(dates_found)

print("Missing dates:")
print(missing_dates)
print("Number of missing dates:", len(missing_dates))

Missing dates:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Number of missing dates: 0


Loading all files.

In [13]:
def read_price_file(file):
    df = pd.read_excel(file, header=None)

    # Find the row where the actual table starts
    header_row = df[df.iloc[:, 0].astype(str).str.contains("Delivery period")].index[0]

    # Set that row as the header
    df.columns = df.iloc[header_row]

    # Remove metadata rows above the header
    df = df[(header_row + 1):]

    # Reset index after slicing
    df = df.reset_index(drop=True)

    return df


prices_dfs = []

for file in price_files:
    try:
        # Read file using custom parser
        df = read_price_file(file)

        # Add source file information for traceability
        df["source_file"] = os.path.basename(file)

        prices_dfs.append(df)

    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all dataframes into a single dataset
prices_raw = pd.concat(prices_dfs, ignore_index=True)

print(f"Combined shape: {prices_raw.shape}")

Combined shape: (19710, 23)


First comprovation of the loaded data.

In [15]:
prices_raw.head()

4,Delivery period (CET),EE (EUR),LT (EUR),LV (EUR),AT (EUR),BE (EUR),FR (EUR),GER (EUR),NL (EUR),DK1 (EUR),...,NO2 (EUR),NO3 (EUR),NO4 (EUR),NO5 (EUR),SE1 (EUR),SE2 (EUR),SE3 (EUR),SE4 (EUR),source_file,PL (EUR)
0,00:00 - 01:00,13.55,13.55,13.55,22.26,22.26,22.26,22.26,22.26,4.25,...,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
1,01:00 - 02:00,7.56,7.56,7.56,20.22,20.22,20.22,20.22,20.22,4.27,...,4.27,4.27,4.27,4.27,4.27,4.27,4.27,4.27,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
2,02:00 - 03:00,7.15,7.15,7.15,19.67,19.67,19.67,19.67,19.67,4.31,...,4.31,4.31,4.31,4.31,4.31,4.31,4.31,4.31,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
3,03:00 - 04:00,8.02,8.02,8.02,19.25,19.25,19.25,19.25,19.25,4.41,...,4.41,4.41,4.41,4.41,4.41,4.41,4.41,4.41,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
4,04:00 - 05:00,7.91,7.91,7.91,19.31,19.31,19.31,19.31,19.31,4.53,...,4.53,4.53,4.53,4.53,4.53,4.53,4.53,4.53,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN


In [17]:
prices_raw.tail()

4,Delivery period (CET),EE (EUR),LT (EUR),LV (EUR),AT (EUR),BE (EUR),FR (EUR),GER (EUR),NL (EUR),DK1 (EUR),...,NO2 (EUR),NO3 (EUR),NO4 (EUR),NO5 (EUR),SE1 (EUR),SE2 (EUR),SE3 (EUR),SE4 (EUR),source_file,PL (EUR)
19705,22:00 - 23:00,69.7,69.7,69.7,105.01,35.96,140.38,5.1,99,32.34,...,139.18,32.34,32.34,139.18,32.34,32.34,32.34,32.34,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19706,23:00 - 00:00,57.98,57.98,57.98,57.98,36.34,127.81,6.32,98.2,29.76,...,138.3,29.76,29.76,138.3,29.76,29.76,29.76,29.76,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19707,Min:,39.11,39.11,39.11,39.11,-40.16,37.33,-0.06,0,5.71,...,112.88,24.01,24.01,112.88,24.01,24.01,24.01,24.01,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19708,Max:,105.07,105.07,105.07,105.07,92.99,193.15,62.23,142.53,105.07,...,147.24,55.36,55.36,147.24,55.36,55.36,105.07,105.07,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19709,Average:,81.49,81.49,81.49,86.96,9.68,120.02,12.13,78.63,42.61,...,135.85,37.9,37.9,135.85,37.9,37.9,43.62,43.62,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN


## **2.2 Volumes**

## **2.3 Flows**

## **2.4 Capacities**

## **3. DATA PREPROCESSING**

## **3.1 Prices**

In this section, the raw Prices dataset is cleaned and transformed in order to make it suitable for exploratory analysis. The preprocessing includes removing summary rows, cleaning column names, extracting the delivery date from the source file, and creating a proper timestamp.

In [30]:
prices = prices_raw.copy()

Next step is deleting unnecessary columns and rows.

In [31]:
# Remove summary rows that do not correspond to actual delivery periods
prices = prices[
    ~prices["Delivery period (CET)"].astype(str).str.contains("Min|Max|Average", na=False)
]

# Reset index after filtering
prices = prices.reset_index(drop=True)

And the following step is clean column names.

In [32]:
# Clean column names by removing the currency suffix
prices.columns = [
    col.replace(" (EUR)", "") if isinstance(col, str) else col
    for col in prices.columns
]

Extract delivery date from the source file name and create a proper timestamp.

In [37]:
# Extract delivery day from the source file name
prices["Delivery day"] = prices["source_file"].str.extract(r"(\d{4}-\d{2}-\d{2})")

# Convert delivery day to datetime
prices["Delivery day"] = pd.to_datetime(prices["Delivery day"])

# Extract the starting hour from the delivery period
prices["hour"] = prices["Delivery period (CET)"].str[:2].astype(int)

# Create a full timestamp combining delivery day and hour
prices["timestamp"] = prices["Delivery day"] + pd.to_timedelta(prices["hour"], unit="h")

# Sort dataset chronologically
prices = prices.sort_values("timestamp").reset_index(drop=True)

In [38]:
prices.head()

,Delivery period (CET),EE,LT,LV,AT,BE,FR,GER,NL,DK1,...,NO5,SE1,SE2,SE3,SE4,source_file,PL,Delivery day,hour,timestamp
0,00:00 - 01:00,28.78,28.78,28.78,41.88,41.88,41.88,41.88,41.88,33.42,...,31.82,28.78,28.78,28.78,28.78,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,0,2020-01-01 00:00:00
1,01:00 - 02:00,28.45,28.45,28.45,38.6,38.6,38.6,38.6,38.6,31.77,...,31.77,28.45,28.45,28.45,28.45,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,1,2020-01-01 01:00:00
2,02:00 - 03:00,27.9,27.9,27.9,36.55,36.55,36.55,36.55,36.55,31.57,...,31.57,27.9,27.9,27.9,27.9,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,2,2020-01-01 02:00:00
3,03:00 - 04:00,27.52,27.52,27.52,32.32,32.32,32.32,32.32,32.32,31.28,...,31.28,27.52,27.52,27.52,27.52,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,3,2020-01-01 03:00:00
4,04:00 - 05:00,27.54,27.54,27.54,30.85,30.85,30.85,30.85,30.85,30.85,...,30.72,27.54,27.54,27.54,27.54,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,4,2020-01-01 04:00:00


In [35]:
prices.columns.tolist()

['Delivery period (CET)',
 'EE',
 'LT',
 'LV',
 'AT',
 'BE',
 'FR',
 'GER',
 'NL',
 'DK1',
 'DK2',
 'FI',
 'NO1',
 'NO2',
 'NO3',
 'NO4',
 'NO5',
 'SE1',
 'SE2',
 'SE3',
 'SE4',
 'source_file',
 'PL',
 'Delivery day',
 'hour',
 'timestamp']

Even we decided to exclude some zones to ensure consistency in the geographical coverage, there are few data from PL we need to remove to avoid inconsistencies in the analysis.

In [41]:
# Drop unwanted zones
prices = prices.drop(columns=["PL"], errors="ignore")

In [43]:
prices.head()

,Delivery period (CET),EE,LT,LV,AT,BE,FR,GER,NL,DK1,...,NO4,NO5,SE1,SE2,SE3,SE4,source_file,Delivery day,hour,timestamp
0,00:00 - 01:00,28.78,28.78,28.78,41.88,41.88,41.88,41.88,41.88,33.42,...,28.78,31.82,28.78,28.78,28.78,28.78,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,0,2020-01-01 00:00:00
1,01:00 - 02:00,28.45,28.45,28.45,38.6,38.6,38.6,38.6,38.6,31.77,...,28.45,31.77,28.45,28.45,28.45,28.45,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,1,2020-01-01 01:00:00
2,02:00 - 03:00,27.9,27.9,27.9,36.55,36.55,36.55,36.55,36.55,31.57,...,27.9,31.57,27.9,27.9,27.9,27.9,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,2,2020-01-01 02:00:00
3,03:00 - 04:00,27.52,27.52,27.52,32.32,32.32,32.32,32.32,32.32,31.28,...,27.52,31.28,27.52,27.52,27.52,27.52,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,3,2020-01-01 03:00:00
4,04:00 - 05:00,27.54,27.54,27.54,30.85,30.85,30.85,30.85,30.85,30.85,...,27.54,30.72,27.54,27.54,27.54,27.54,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,4,2020-01-01 04:00:00


Last verification of the preprocessed data.

In [49]:
prices.shape
prices[["timestamp", "Delivery day", "Delivery period (CET)"]].head()
prices.isna().sum().sort_values(ascending=False).head(10)

Delivery period (CET)    0
NO2                      0
hour                     0
Delivery day             0
source_file              0
SE4                      0
SE3                      0
SE2                      0
SE1                      0
NO5                      0
dtype: int64

## **3.2 Volumes**

## **3.3 Flows**

## **3.4 Capacities**

## **4. EXPLORATORY DATA ANALYSIS (EDA)**